In [1]:
# Load libraries
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import statsmodels.api as sm

sns.set_style('darkgrid')
import math
from sklearn.linear_model import LogisticRegression
from scipy import stats

import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

from license_model import LicenseModel

# 1.0 Setup and run the model

### 1.1 Setup

In [2]:
# INPUT VARIABLES

CITY = 'Toronto'
INPUT_FILE = os.path.join('data', 'toronto_inside_airbnb_data', 'may_3_2025.csv')
MIN_NIGHTS_FOR_LICENSE = 28
LICENSE_REGEX_PATTERN = r'^STR-\d{4}-[A-Za-z]{6}$'

In [7]:
city_model = LicenseModel(
    input_file=INPUT_FILE,
    city=CITY,
    min_nights_for_license=MIN_NIGHTS_FOR_LICENSE,
    license_regex_pattern=LICENSE_REGEX_PATTERN
)

# city_model.run_initial_setup(print_info=False)

In [8]:
city_model.load_data()

In [9]:
city_model.preprocess_data(print_info=True)

Columns with constant values: []


In [10]:
city_model.listings_df

,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
17,within a few hours,100%,87%,t,1.0,1.0,"['email', 'phone']",t,t,Niagara,...,4.97,4.98,4.92,STR-2009-FXRRPD,f,1,1,0,0,1.19
22,within an hour,83%,90%,f,1.0,2.0,"['email', 'phone']",t,t,Waterfront Communities-The Island,...,4.61,4.88,4.37,STR-2408-JCKSBZ,f,1,1,0,0,0.32
29,within an hour,100%,100%,t,1.0,1.0,"['email', 'phone']",t,t,Oakridge,...,4.98,4.93,4.89,STR-2012-HLWPHK,f,1,1,0,0,0.58
32,within an hour,100%,63%,t,3.0,5.0,"['email', 'phone', 'work_email']",t,t,Cabbagetown-South St.James Town,...,4.92,4.82,4.86,STR-2104-FLPKVB,f,3,3,0,0,0.52
39,within a few hours,100%,52%,t,1.0,1.0,"['email', 'phone']",t,t,North St.James Town,...,4.78,4.81,4.84,STR-2009-GXRRPH,f,1,1,0,0,0.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21382,within an hour,100%,92%,t,3.0,3.0,['phone'],t,t,Waterfront Communities-The Island,...,NaN,NaN,NaN,STR-2410-HPKVPX,f,3,0,3,0,NaN
21383,NaN,NaN,75%,f,2.0,14.0,"['email', 'phone']",t,t,Hillcrest Village,...,NaN,NaN,NaN,STR-2406-FDVBPP,t,2,2,0,0,NaN
21386,within an hour,100%,94%,f,21.0,24.0,['phone'],t,t,L'Amoreaux,...,NaN,NaN,NaN,STR-2303-HSCKVY,f,8,0,8,0,NaN
21389,within an hour,100%,100%,t,3.0,6.0,"['email', 'phone']",t,t,Waterfront Communities-The Island,...,NaN,NaN,NaN,STR-2312-HQFHHX,f,2,2,0,0,NaN


In [14]:
# How many rows have nans?
print(f"Number of rows with NaN values: {city_model.listings_df.isna().sum().sum()}")

# From the rows that have NaN values in any column, how many have 'unknown' in the 'license' column?
city_model.listings_df[city_model.listings_df.isna().any(axis=1)]['license'].str.contains('unknown', na=False).sum()

Number of rows with NaN values: 8428


np.int64(41)

### 1.2 Check assumptions

In [4]:
city_model.listings_df

,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,bathrooms,bedrooms,beds,price,minimum_nights,availability_30,...,neighbourhood_cleansed_Woodbine-Lumsden,neighbourhood_cleansed_Wychwood,neighbourhood_cleansed_Yonge-Eglinton,neighbourhood_cleansed_Yonge-St.Clair,neighbourhood_cleansed_York University Heights,neighbourhood_cleansed_Yorkdale-Glen Park,room_type_Hotel room,room_type_Private room,room_type_Shared room,legal_listing
22,83.0,90.0,0.0,2.0,2.0,2.0,2.0,269.0,2.0,23.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
29,100.0,100.0,1.0,1.0,1.0,1.0,1.0,92.0,18.0,29.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
39,100.0,52.0,1.0,1.0,1.5,2.0,3.0,240.0,4.0,24.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
56,100.0,100.0,0.0,3.0,1.0,1.0,1.0,254.0,14.0,23.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
61,100.0,38.0,0.0,1.0,1.0,1.0,1.0,287.0,3.0,23.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21246,100.0,96.0,0.0,3.0,1.0,1.0,2.0,144.0,1.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
21262,100.0,100.0,0.0,1.0,1.0,2.0,3.0,217.0,1.0,25.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
21267,100.0,100.0,0.0,2.0,1.0,1.0,1.0,86.0,1.0,24.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
21274,50.0,100.0,0.0,6.0,1.0,2.0,1.0,145.0,1.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [5]:
violating_vars = city_model.check_linearity_of_independent_variables_and_log_odds()
print("Significant variables (p < 0.05) that violate linearity assumption:")
print(violating_vars)

Significant variables (p < 0.05) that violate linearity assumption:
[]


In [ ]:
delete_instead_of_transform = True  # Set to True if you want to delete the violating variables instead of transforming them

if delete_instead_of_transform:
    # If we are deleting the violating variables, we will remove them from the DataFrame
    for var in violating_vars:
        print(f"Deleting variable: {var}")
        city_model.listings_df.drop(columns=[var], inplace=True)
else:
    print("Transforming variables to resolve linearity issues...")

    # Test multiple transformations for each violating variable
    for var in violating_vars:
        print("_" * 50)
        # Make a clean copy of the original data
        listings_data_temp = city_model.listings_df.copy()

        print(f"Testing transformations for variable: {var}")

        # Define a list of transformations to try
        transformations = {
            "log": lambda x: np.log1p(x),  # log(1 + x), handles zero values
            "sqrt": lambda x: np.sqrt(x),  # square root
            "reciprocal": lambda x: 1 / (x + 1e-6),  # reciprocal, avoids division by zero
            "square": lambda x: x**2,  # square
            "cube": lambda x: x**3,  # cube
            "exp": lambda x: np.exp(x),  # exponential
            "inverse_sqrt": lambda x: 1 / (np.sqrt(x) + 1e-6),  # inverse square root
            "log10": lambda x: np.log10(x + 1e-6),  # base-10 logarithm
            "log2": lambda x: np.log2(x + 1e-6),  # base-2 logarithm
            "zscore": lambda x: (x - np.mean(x)) / np.std(x),  # z-score normalization
            "minmax": lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)),  # min-max scaling
            "sigmoid": lambda x: 1 / (1 + np.exp(-x)),  # sigmoid transformation
            "tanh": lambda x: np.tanh(x),  # hyperbolic tangent
            "arcsinh": lambda x: np.arcsinh(x),  # inverse hyperbolic sine
            "cbrt": lambda x: np.cbrt(x),  # cube root
            "abs": lambda x: np.abs(x),  # absolute value
            "clip": lambda x: np.clip(x, 0, np.percentile(x, 95)),  # clip outliers at 95th percentile
            "rank": lambda x: x.rank(method="average"),  # rank transformation
            "binarize": lambda x: (x > np.median(x)).astype(int),  # binarize based on median
        }

        # Try each transformation
        for name, transform in transformations.items():
            try:
                # Apply the transformation
                transformed_var = transform(listings_data_temp[var])
                listings_data_temp[var] = transformed_var

                # Re-check linearity with the transformed variable
                new_violating_vars = city_model.check_linearity_of_independent_variables_and_log_odds(
                    df_city_1=listings_data_temp
                )

                if var not in new_violating_vars:
                    print(f"Transformation '{name}' resolved linearity issue for {var}.")

                    # Update the model with the transformed variable
                    city_model.listings_df[f"{var}_{name}"] = transformed_var
                    city_model.listings_df.drop(columns=[var], inplace=True)
                    
                    break  # Stop testing further transformations for this variable
                else:
                    # print(f"Transformation '{name}' did not resolve linearity issue for {var}.")
                    pass
            except Exception as e:
                print(f"Error applying transformation '{name}' to {var}: {e}")

        else:
            # If no transformation resolved the issue
            print(f"No transformation resolved the linearity issue for {var}.")

In [6]:
city_model.check_no_strongly_influential_outliers(print_info=True, show_plot=True, remove_hi_outliers=True)

Optimization terminated successfully.
         Current function value: 0.013083
         Iterations: 504
         Function evaluations: 527
         Gradient evaluations: 518


ValueError: need covariance of parameters for computing (unnormalized) covariances

### 1.3 Run Model

In [ ]:
city_model.train_model()

In [ ]:
city_model.evaluate_model(selected_threshold=0.5)

In [ ]:
city_model.does_data_follow_one_in_ten_rule()

In [ ]:
city_model.logit_model.create_roc_curve()